# PFA - Reconnaissance des Emotions Faciales
## Etape 4c : Deep Learning - ResNet50 (Transfer Learning)

**Equipe :** Ayman TEMSAMANI, Nouhaila AIT ABDELAH, Rabab ZITI  
**Encadrant :** Pr. Redouane DAHBI  
**Date :** Semaine S7-S11 (Phase 4 du planning)

**Contexte :** Conformement au cahier des charges (Section 5.2), cette etape implemente ResNet50 avec transfer learning. L'objectif est d'atteindre 70-75% de precision en exploitant des poids pre-entraines sur ImageNet (1.2 million d'images, 1000 classes).

---
### Justification theorique des choix architecturaux

**Pourquoi ResNet50 ?**

ResNet (Residual Network) introduit les connexions residuelles (skip connections) qui resolvent le probleme du vanishing gradient dans les reseaux profonds. Au lieu d'apprendre la fonction H(x), le reseau apprend le residu F(x) = H(x) - x. Cela permet d'entrainer des reseaux de 50, 101, voire 152 couches sans degradation des performances.

**Pourquoi le transfer learning ?**

FER2013 ne contient que 28 709 images d'entrainement. Entrainer un reseau profond from scratch sur si peu de donnees conduit inevitablement a l'overfitting. Le transfer learning exploite les representations apprises sur ImageNet (1.2M images, 1000 classes) : les premieres couches detectent des features universelles (contours, textures, formes) qui sont reutilisables pour la reconnaissance d'emotions.

**Pourquoi geler les couches en Phase 1 ?**

Si on entraine toutes les couches des le debut, les gradients aleatoires du classifieur non-initialise detruiraient les representations pre-entrainees. On gele d'abord le backbone, on entraine le classifieur jusqu'a convergence, puis on degele progressivement pour le fine-tuning.

**Pourquoi GlobalAveragePooling2D au lieu de Flatten ?**

Flatten transformerait le feature map 7x7x2048 en un vecteur de 100 352 parametres, causant un overfitting massif. GlobalAveragePooling2D reduit chaque canal a sa moyenne, produisant seulement 2048 valeurs, ce qui agit comme un regularisateur structurel.

**Pourquoi 224x224 au lieu de 48x48 ?**

ResNet50 a ete concu et entraine sur des images 224x224. Les poids pre-entraines sont optimises pour cette resolution. Reduire a 48x48 degraderait les representations apprises. On upsample donc les images FER2013 de 48x48 a 224x224.

**Pourquoi RGB au lieu de grayscale ?**

ResNet50 attend 3 canaux en entree (RGB). On convertit les images grayscale en RGB en repetant le canal 3 fois. Ceci est gere automatiquement par flow_from_directory avec color_mode='rgb'.

**Pourquoi Adam avec lr=0.001 en Phase 1 puis 0.0001 en Phase 2 ?**

En Phase 1, le classifieur est aleatoire : un learning rate eleve (0.001) permet une convergence rapide. En Phase 2 (fine-tuning), on ajuste des poids deja bons : un learning rate faible (0.0001) evite de detruire les representations apprises.

In [ ]:
# ============================================================
#  SECTION 1 : Import des bibliotheques
# ============================================================

# numpy : manipulation de tableaux et calculs numeriques.
# Utilise pour argmax (conversion probabilites -> classe) et concatenation.
import numpy as np

# matplotlib.pyplot : tracer les courbes d'apprentissage (accuracy/loss).
# Essentiel pour diagnostiquer overfitting et convergence.
import matplotlib.pyplot as plt

# seaborn : visualisation avancee de la matrice de confusion.
# heatmap() produit une matrice lisible avec echelle de couleurs.
import seaborn as sns

# time : mesurer la duree d'entrainement et le temps d'inference.
# Necessaire pour verifier la contrainte temps reel (>15 FPS).
import time

# tensorflow : framework de Deep Learning.
import tensorflow as tf

# ResNet50 : architecture pre-entrainee sur ImageNet.
# include_top=False retire les couches fully-connected originales (1000 classes).
# weights='imagenet' charge les poids pre-entraines.
from tensorflow.keras.applications import ResNet50

# Model : API fonctionnelle Keras pour construire le modele.
from tensorflow.keras.models import Model

# Couches du classifieur personnalise.
# GlobalAveragePooling2D : reduit le feature map 3D en vecteur 1D par moyenne.
#   Avantage sur Flatten : moins de parametres, anti-overfitting.
# Dense : couche fully-connected.
# Dropout : desactive aleatoirement des neurones pour eviter l'overfitting.
# BatchNormalization : normalise les activations pour stabiliser l'apprentissage.
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D, BatchNormalization

# Adam : optimiseur avec taux d'apprentissage adaptatif par parametre.
# Combine Momentum et RMSprop. Standard pour la classification d'images.
from tensorflow.keras.optimizers import Adam

# Callbacks : fonctions executees pendant l'entrainement.
# ReduceLROnPlateau : reduit le learning rate si la val_loss stagne.
# EarlyStopping : arrete l'entrainement si pas d'amelioration.
# ModelCheckpoint : sauvegarde le meilleur modele.
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping, ModelCheckpoint

# ImageDataGenerator : chargement et augmentation des images en temps reel.
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Metriques d'evaluation conformes au cahier des charges (Section 5.4).
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score

print("=" * 60)
print("BIBLIOTHEQUES IMPORTEES")
print("=" * 60)
print(f"TensorFlow version : {tf.__version__}")
print(f"GPU disponible      : {tf.config.list_physical_devices('GPU')}")

In [ ]:
# ============================================================
#  SECTION 2 : Chargement des donnees
# ============================================================

# Chemins vers les donnees FER2013.
# Les dossiers train/ et test/ contiennent chacun 7 sous-dossiers
# correspondant aux 7 classes d'emotions.
TRAIN_DIR = 'C:/Users/ayman/Desktop/PFA/archive/train'
TEST_DIR  = 'C:/Users/ayman/Desktop/PFA/archive/test'

# ResNet50 a ete entraine sur des images 224x224 RGB.
# On upsample les images FER2013 de 48x48 a 224x224.
# Pourquoi ne pas garder 48x48 ?
#   - Les poids pre-entraines sont optimises pour 224x224.
#   - A 48x48, les features apprises sur ImageNet seraient degradees.
#   - L'upsampling est gere par flow_from_directory (target_size).
IMG_SIZE = 224

# Batch size reduit a 32 (au lieu de 64 pour le CNN custom).
# Pourquoi ?
#   - Les images 224x224 sont 21x plus lourdes que 48x48.
#   - ResNet50 a 25M de parametres.
#   - 32 images/batch tient dans les 4 Go de VRAM du GPU.
BATCH_SIZE = 32

print("=" * 60)
print("CONFIGURATION DES GENERATEURS")
print("=" * 60)
print(f"Taille d'entree : {IMG_SIZE}x{IMG_SIZE} RGB")
print(f"Batch size      : {BATCH_SIZE}")

# Generateur d'entrainement AVEC data augmentation.
# Memes parametres que l'etape 3, conformes au cahier des charges.
# Pourquoi l'augmentation est cruciale pour ResNet50 ?
#   - ResNet50 a 25M de parametres, risque d'overfitting eleve.
#   - L'augmentation cree des variantes infinies, empechant la memorisation.
#   - Rotation : robustesse a l'inclinaison de tete.
#   - Decalages : robustesse au cadrage imparfait.
#   - Zoom : robustesse a la distance variable.
#   - Miroir : symetrie des expressions faciales.
#   - Luminosite : robustesse aux conditions d'eclairage.
train_datagen = ImageDataGenerator(
    rescale=1./255,              # Normalisation [0, 1] obligatoire
    rotation_range=15,           # ±15 degres
    width_shift_range=0.1,       # Decalage horizontal 10%
    height_shift_range=0.1,      # Decalage vertical 10%
    zoom_range=0.1,              # Zoom ×0.9 a ×1.1
    horizontal_flip=True,        # Miroir horizontal
    brightness_range=[0.8, 1.2], # Variation de luminosite
    fill_mode='nearest'          # Remplissage des pixels vides
)

# Generateur de test SANS augmentation.
# Principe fondamental : evaluer sur des images reelles, non modifiees.
# Seule la normalisation est appliquee.
test_datagen = ImageDataGenerator(rescale=1./255)

# Chargement des images d'entrainement.
# flow_from_directory() :
#   - Lit les images depuis les sous-dossiers.
#   - Redimensionne a target_size.
#   - Convertit en RGB (color_mode='rgb').
#   - Attribue les labels automatiquement.
#   - Applique l'augmentation en temps reel.
train_generator = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),  # 224x224 pour ResNet50
    color_mode='rgb',                   # 3 canaux pour ResNet50
    class_mode='categorical',           # One-hot encoding (7 classes)
    batch_size=BATCH_SIZE,
    shuffle=True                        # Melanger a chaque epoque
)

# Chargement des images de test.
# shuffle=False : ordre fixe pour reproductibilite.
test_generator = test_datagen.flow_from_directory(
    TEST_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    color_mode='rgb',
    class_mode='categorical',
    batch_size=BATCH_SIZE,
    shuffle=False
)

# Affichage des informations de chargement.
print(f"\nClasses detectees : {train_generator.class_indices}")
print(f"Images d'entrainement : {train_generator.samples}")
print(f"Images de test        : {test_generator.samples}")
print(f"Batchs par epoque     : {len(train_generator)}")

In [ ]:
# ============================================================
#  SECTION 3 : Construction du modele ResNet50 + Classifieur
# ============================================================

print("=" * 60)
print("CONSTRUCTION DU MODELE RESNET50")
print("=" * 60)

# Chargement de ResNet50 sans les couches fully-connected (include_top=False).
# Le modele original a ete entraine sur ImageNet (1000 classes).
# On retire la tete pour ajouter notre propre classifieur 7 emotions.
#
# Pourquoi include_top=False ?
#   - Le top original classifie 1000 classes (pas nos 7 emotions).
#   - On veut des features generiques, pas des predictions ImageNet.
#
# Pourquoi weights='imagenet' ?
#   - Poids pre-entraines sur 1.2M images pendant des semaines.
#   - Represente des millions d'heures de calcul qu'on reutilise gratuitement.
#   - Sans ces poids, ResNet50 sur FER2013 donnerait ~40% (pire que SVM).
#
# Pourquoi input_shape=(224,224,3) ?
#   - Format standard de ResNet50.
#   - 3 canaux = RGB. Les images grayscale sont automatiquement converties.
base_model = ResNet50(
    include_top=False,
    weights='imagenet',
    input_shape=(IMG_SIZE, IMG_SIZE, 3)
)

print(f"Backbone ResNet50 charge.")
print(f"Nombre de couches totales : {len(base_model.layers)}")

# Gel de TOUTES les couches du backbone.
# layer.trainable = False empeche la mise a jour des poids pendant
# la retropropagation. Ces couches agissent comme un extracteur de features fixe.
#
# Pourquoi geler ?
#   - Les poids pre-entraines sont deja excellents pour detecter formes, textures.
#   - Au debut, le classifieur est aleatoire : ses gradients detruiraient
#     les bonnes representations du backbone.
#   - On entraine d'abord le classifieur, puis on degelera en Phase 2.
for layer in base_model.layers:
    layer.trainable = False

print(f"Toutes les couches du backbone sont gelees.")
print(f"Couches trainables (backbone) : {sum(1 for l in base_model.layers if l.trainable)}")

# Construction du classifieur personnalise.
# On utilise l'API fonctionnelle (Model) au lieu de Sequential.
# Cela permet de connecter le backbone ResNet50 a nos couches.

# Entree : sortie du backbone ResNet50.
# La sortie de base_model est un feature map 7x7x2048.
# 7x7 = dimensions spatiales (224/32 = 7 apres 5 pooling).
# 2048 = nombre de filtres de la derniere couche convolutive.
x = base_model.output

# GlobalAveragePooling2D : moyenne spatiale de chaque canal.
# Transforme (7, 7, 2048) en (2048,).
#
# Pourquoi GAP et pas Flatten ?
#   - Flatten produirait 7*7*2048 = 100 352 parametres → overfitting.
#   - GAP produit 2048 parametres → regularisation forte.
#   - GAP est invariant a la position : utile si le visage est decale.
x = GlobalAveragePooling2D(name='global_avg_pool')(x)

# Dense 512 : premiere couche fully-connected.
# Activation relu : introduit la non-linearite.
# Pourquoi 512 neurones ?
#   - Assez grand pour apprendre des combinaisons complexes de features.
#   - Assez petit pour ne pas causer d'overfitting.
#   - Valeur standard dans la litterature pour FER.
x = Dense(512, activation='relu', name='fc1')(x)

# BatchNormalization : normalise les activations du batch.
# Pourquoi ?
#   - Stabilise l'apprentissage (reduit la sensibilite a l'initialisation).
#   - Permet un learning rate plus eleve.
#   - Agit comme un regularisateur leger (reduit l'overfitting).
x = BatchNormalization(name='bn1')(x)

# Dropout 0.5 : desactive 50% des neurones aleatoirement.
# Pourquoi 0.5 ?
#   - 512 neurones, c'est beaucoup de parametres.
#   - 0.5 est le taux optimal pour les couches denses (Hinton et al., 2012).
#   - Force le reseau a apprendre des representations redondantes et robustes.
x = Dropout(0.5, name='dropout1')(x)

# Dense 256 : deuxieme couche fully-connected.
# Reduction progressive : 2048 → 512 → 256 → 7.
# Cette architecture en entonnoir est standard pour la classification.
x = Dense(256, activation='relu', name='fc2')(x)
x = BatchNormalization(name='bn2')(x)
x = Dropout(0.5, name='dropout2')(x)

# Couche de sortie : 7 neurones, activation softmax.
# Softmax transforme les logits en probabilites (somme = 1).
# Chaque neurone correspond a une classe d'emotion.
output = Dense(7, activation='softmax', name='output')(x)

# Creation du modele complet.
# Model(inputs, outputs) connecte l'entree de ResNet50 a notre sortie.
model = Model(inputs=base_model.input, outputs=output)

print("\n" + "=" * 60)
print("ARCHITECTURE COMPLETE")
print("=" * 60)
model.summary()

In [ ]:
# ============================================================
#  SECTION 4 : Compilation du modele
# ============================================================

# Compilation definit :
#   - L'optimiseur (comment mettre a jour les poids).
#   - La fonction de perte (comment mesurer l'erreur).
#   - Les metriques (comment evaluer la performance).

# Optimiseur : Adam avec learning_rate=0.001.
# Pourquoi Adam ?
#   - Taux d'apprentissage adaptatif par parametre.
#   - Combine les avantages de Momentum (stable) et RMSprop (rapide).
#   - Standard de facto pour la classification d'images.
# Pourquoi lr=0.001 ?
#   - Valeur par defaut eprouvee pour le transfer learning.
#   - Assez eleve pour converger rapidement en Phase 1.
#   - Sera reduit en Phase 2 (fine-tuning).

# Loss : categorical_crossentropy.
# Pourquoi ?
#   - Classification multi-classe (>2 classes).
#   - Compatible avec les labels one-hot encodes.
#   - Mesure la dissimilarite entre distribution predite et distribution reelle.

# Metrique : accuracy.
# Pourquoi ?
#   - Facile a interpreter (pourcentage de predictions correctes).
#   - Exigee par le cahier des charges (Section 5.4).

model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print("=" * 60)
print("MODELE COMPILE")
print("=" * 60)
print(f"Optimizer : Adam (lr=0.001)")
print(f"Loss      : categorical_crossentropy")
print(f"Metrics   : accuracy")

# Verification du gel.
trainable_count = sum(1 for l in model.layers if l.trainable)
non_trainable_count = sum(1 for l in model.layers if not l.trainable)
print(f"\nCouches trainables    : {trainable_count}")
print(f"Couches non-trainables : {non_trainable_count}")
print(f"\nSeules les couches du classifieur seront entrainees en Phase 1.")

In [ ]:
# ============================================================
#  SECTION 5 : Configuration des callbacks
# ============================================================

# Callback 1 : ReduceLROnPlateau
# Reduit le learning rate quand la val_loss cesse de s'ameliorer.
#
# Parametres :
#   - monitor='val_loss' : surveille la perte de validation.
#   - factor=0.5 : divise le lr par 2 a chaque declenchement.
#   - patience=5 : attend 5 epoques sans amelioration avant de reduire.
#   - min_lr=1e-6 : ne descend pas en dessous de ce seuil.
#
# Pourquoi c'est utile ?
#   - Si le modele plafonne, un lr plus petit permet des ajustements fins.
#   - Evite les oscillations autour du minimum.
reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=5,
    min_lr=1e-6,
    verbose=1
)

# Callback 2 : EarlyStopping
# Arrete l'entrainement si la val_accuracy ne s'ameliore plus.
#
# Parametres :
#   - monitor='val_accuracy' : surveille la precision de validation.
#   - patience=12 : attend 12 epoques sans amelioration.
#   - restore_best_weights=True : restaure les meilleurs poids a l'arret.
#
# Pourquoi patience=12 ?
#   - Assez long pour laisser le modele explorer.
#   - Assez court pour ne pas perdre des heures.
#   - Le fine-tuning peut stagner puis rebondir.
early_stop = EarlyStopping(
    monitor='val_accuracy',
    patience=12,
    restore_best_weights=True,
    verbose=1
)

# Callback 3 : ModelCheckpoint
# Sauvegarde le modele a chaque fois que val_accuracy s'ameliore.
#
# Parametres :
#   - filepath : chemin du fichier de sauvegarde.
#   - monitor='val_accuracy' : sauvegarde si meilleure val_accuracy.
#   - save_best_only=True : ecrase le fichier precedent.
#   - verbose=1 : affiche un message a chaque sauvegarde.
checkpoint = ModelCheckpoint(
    'best_resnet50_model.h5',
    monitor='val_accuracy',
    save_best_only=True,
    verbose=1
)

callbacks = [reduce_lr, early_stop, checkpoint]

print("=" * 60)
print("CALLBACKS CONFIGURES")
print("=" * 60)
print("1. ReduceLROnPlateau  : divise lr par 2 apres 5 epoques sans amelioration")
print("2. EarlyStopping      : arrete apres 12 epoques sans amelioration")
print("3. ModelCheckpoint    : sauvegarde le meilleur modele")

In [ ]:
# ============================================================
#  SECTION 6 : Entrainement Phase 1 (Classifieur uniquement)
# ============================================================

print("=" * 60)
print("PHASE 1 : ENTRAINEMENT DU CLASSIFIEUR")
print("=" * 60)
print("Backbone ResNet50 : GELE (extracteur de features fixe)")
print("Classifieur        : ENTRAINE (poids aleatoires -> optimaux)")
print("Learning rate      : 0.001")
print("Duree estimee      : 2-4 heures")
print("=" * 60)

# Nombre d'epoques pour la Phase 1.
# 30 epoques est un bon compromis :
#   - Assez pour converger (le classifieur part de zero).
#   - Pas trop pour eviter l'overfitting.
#   - EarlyStopping arrettera probablement avant.
EPOCHS_PHASE1 = 30

start_time = time.time()

# model.fit() : lance l'entrainement.
#
# Parametres :
#   - train_generator : flux infini d'images augmentees.
#   - steps_per_epoch : nombre de batchs par epoque (taille dataset / batch_size).
#   - epochs : nombre de passages complets sur les donnees.
#   - validation_data : donnees de test pour evaluer a chaque epoque.
#   - validation_steps : nombre de batchs de validation.
#   - callbacks : fonctions executees pendant l'entrainement.
history = model.fit(
    train_generator,
    steps_per_epoch=len(train_generator),
    epochs=EPOCHS_PHASE1,
    validation_data=test_generator,
    validation_steps=len(test_generator),
    callbacks=callbacks,
    verbose=1
)

phase1_time = time.time() - start_time
print(f"\nPhase 1 terminee en {phase1_time/60:.1f} minutes.")
print(f"Meilleure val_accuracy Phase 1 : {max(history.history['val_accuracy'])*100:.2f}%")

In [ ]:
# ============================================================
#  SECTION 7 : Phase 2 - Fine-Tuning
# ============================================================

print("=" * 60)
print("PHASE 2 : FINE-TUNING")
print("=" * 60)

# Le fine-tuning consiste a degeler les dernieres couches du backbone
# pour les adapter specifiquement a notre tache (emotions faciales).
#
# Pourquoi degeler seulement les 50 dernieres couches ?
#   - Les premieres couches detectent des features universelles
#     (contours, couleurs) qui sont identiques pour toutes les taches.
#   - Les dernieres couches apprennent des features specifiques
#     (formes d'yeux, textures de peau) qu'on veut adapter aux emotions.
#   - Degeler toutes les couches avec peu de donnees = overfitting.
#   - 50 est un nombre standard dans la litterature.

print("Degelage des 50 dernieres couches du backbone...")

for layer in base_model.layers[-50:]:
    layer.trainable = True

trainable_now = sum(1 for l in model.layers if l.trainable)
non_trainable_now = sum(1 for l in model.layers if not l.trainable)

print(f"Couches trainables    : {trainable_now}")
print(f"Couches non-trainables : {non_trainable_now}")

# Recompilation avec un learning rate plus faible.
#
# Pourquoi lr=0.0001 (10x plus petit) ?
#   - Les poids du backbone sont deja bons.
#   - Un grand lr detruirait les representations pre-entraines.
#   - On fait des ajustements fins, pas des changements radicaux.
#   - 0.0001 = 1e-4 est la valeur standard pour le fine-tuning.
model.compile(
    optimizer=Adam(learning_rate=0.0001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print(f"\nModele recompile avec lr=0.0001 pour le fine-tuning.")

# Nombre d'epoques pour le fine-tuning.
# 20 epoques suffisent generalement :
#   - Le modele est deja bon apres la Phase 1.
#   - Les ajustements sont subtils.
#   - Trop d'epoques → overfitting.
EPOCHS_PHASE2 = 20

print(f"\nDebut du fine-tuning ({EPOCHS_PHASE2} epoques max)...")

history_ft = model.fit(
    train_generator,
    steps_per_epoch=len(train_generator),
    epochs=EPOCHS_PHASE2,
    validation_data=test_generator,
    validation_steps=len(test_generator),
    callbacks=callbacks,
    verbose=1
)

total_time = (time.time() - start_time) / 60
print(f"\nFine-tuning termine en {total_time:.1f} minutes (total).")
print(f"Meilleure val_accuracy Phase 2 : {max(history_ft.history['val_accuracy'])*100:.2f}%")

In [ ]:
# ============================================================
#  SECTION 8 : Courbes d'apprentissage
# ============================================================

print("=" * 60)
print("COURBES D'APPRENTISSAGE")
print("=" * 60)

# Les courbes montrent l'evolution de l'accuracy et de la loss
# pendant les deux phases d'entrainement.
#
# Ce qu'on cherche :
#   - Train accuracy qui augmente → le modele apprend.
#   - Val accuracy qui suit → bonne generalisation.
#   - Train >> Val → overfitting (probleme).
#   - Val qui plafonne → modele a atteint sa capacite.

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Graphique 1 : Accuracy
axes[0].plot(history.history['accuracy'], label='Train Phase 1', color='blue', linestyle='--')
axes[0].plot(history.history['val_accuracy'], label='Val Phase 1', color='blue', linestyle='-')
axes[0].plot(history_ft.history['accuracy'], label='Train Phase 2', color='red', linestyle='--')
axes[0].plot(history_ft.history['val_accuracy'], label='Val Phase 2', color='red', linestyle='-')
axes[0].set_title('Accuracy ResNet50 - Phase 1 + Fine-Tuning', fontsize=14)
axes[0].set_xlabel('Epoque')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Graphique 2 : Loss
axes[1].plot(history.history['loss'], label='Train Phase 1', color='blue', linestyle='--')
axes[1].plot(history.history['val_loss'], label='Val Phase 1', color='blue', linestyle='-')
axes[1].plot(history_ft.history['loss'], label='Train Phase 2', color='red', linestyle='--')
axes[1].plot(history_ft.history['val_loss'], label='Val Phase 2', color='red', linestyle='-')
axes[1].set_title('Loss ResNet50 - Phase 1 + Fine-Tuning', fontsize=14)
axes[1].set_xlabel('Epoque')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
#  SECTION 9 : Evaluation sur le jeu de test
# ============================================================

print("=" * 60)
print("EVALUATION RESNET50 SUR LE JEU DE TEST")
print("=" * 60)

# Reset du generateur pour repartir du debut.
test_generator.reset()

# Prediction sur tout le jeu de test.
# predict() retourne des probabilites (softmax).
# np.argmax() convertit en classe predite.
y_pred_probs = model.predict(test_generator, steps=len(test_generator))
y_pred = np.argmax(y_pred_probs, axis=1)

# y_true est stocke dans test_generator.classes.
# Il contient les vrais labels (indices 0-6).
y_true = test_generator.classes

# Calcul des metriques exigees par le cahier des charges (Section 5.4).
accuracy = accuracy_score(y_true, y_pred)
f1_macro = f1_score(y_true, y_pred, average='macro')

emotion_names = ['Angry', 'Disgust', 'Fear', 'Happy', 'Neutral', 'Sad', 'Surprise']

print(f"\n{'='*40}")
print(f"Accuracy  : {accuracy:.4f} ({accuracy*100:.2f}%)")
print(f"F1-Score (macro) : {f1_macro:.4f}")
print(f"{'='*40}")

# Rapport de classification detaille par classe.
# Precision : parmi les predictions de cette classe, combien sont correctes.
# Recall    : parmi les vrais exemples de cette classe, combien sont detectes.
# F1-score  : moyenne harmonique de precision et recall.
print("\nRapport de classification :")
print(classification_report(y_true, y_pred, target_names=emotion_names, zero_division=0))

In [ ]:
# ============================================================
#  SECTION 10 : Matrice de confusion
# ============================================================

print("=" * 60)
print("MATRICE DE CONFUSION")
print("=" * 60)

# La matrice de confusion montre ou le modele se trompe.
# Lignes = emotions reelles.
# Colonnes = emotions predites.
# Diagonale = predictions correctes.
# Hors-diagonale = confusions.

cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(10, 8))
sns.heatmap(
    cm,
    annot=True,           # Afficher les valeurs numeriques
    fmt='d',              # Format entier (decimal)
    cmap='Blues',         # Echelle de couleur bleue
    xticklabels=emotion_names,
    yticklabels=emotion_names
)
plt.title('Matrice de Confusion - ResNet50', fontsize=16)
plt.xlabel('Emotion predite', fontsize=12)
plt.ylabel('Emotion reelle', fontsize=12)
plt.tight_layout()
plt.show()

# Analyse des confusions principales.
print("\nConfusions principales attendues :")
print("  - Fear ↔ Surprise : expressions proches (yeux ecarquilles)")
print("  - Angry ↔ Disgust : sourcils fronces dans les deux cas")
print("  - Sad ↔ Neutral  : expressions peu intenses")

In [ ]:
# ============================================================
#  SECTION 11 : Temps d'inference
# ============================================================

print("=" * 60)
print("MESURE DU TEMPS D'INFERENCE")
print("=" * 60)

# Le cahier des charges (Section 5.4) exige le temps d'inference par image.
# On mesure sur un batch de 32 images, 100 fois, et on prend la moyenne.

test_generator.reset()
batch = next(test_generator)

n_runs = 100
start = time.time()
for _ in range(n_runs):
    _ = model.predict(batch[0], verbose=0)
elapsed_ms = (time.time() - start) / n_runs * 1000

time_per_image = elapsed_ms / BATCH_SIZE
fps = 1000 / time_per_image

print(f"\nTemps d'inference par batch de {BATCH_SIZE} : {elapsed_ms:.2f} ms")
print(f"Temps d'inference par image              : {time_per_image:.2f} ms")
print(f"FPS equivalent                           : {fps:.1f}")
print(f"\nObjectif cahier des charges : > 15 FPS")

if fps >= 15:
    print("✅ ResNet50 respecte la contrainte temps reel.")
else:
    print("⚠️ ResNet50 est plus lent. Utiliser un modele optimise pour le temps reel.")
    print("   Solutions : TensorRT, ONNX, ou modele plus leger (MobileNet).")

In [ ]:
# ============================================================
#  SECTION 12 : Sauvegarde du modele
# ============================================================

print("=" * 60)
print("SAUVEGARDE DU MODELE")
print("=" * 60)

# Sauvegarde du modele complet (architecture + poids).
# Format HDF5 (.h5) : standard, compatible avec Keras/TensorFlow.
model.save('resnet50_emotion_model.h5')

print("\nFichiers sauvegardes :")
print("  - resnet50_emotion_model.h5 : modele final")
print("  - best_resnet50_model.h5    : meilleur modele (checkpoint)")
print("\nCes modeles seront utilises pour :")
print("  1. Comparaison avec SVM et CNN Custom")
print("  2. Ensemble avec VGG16")
print("  3. Interface temps reel (webcam)")

In [ ]:
# ============================================================
#  SECTION 13 : Comparaison finale des modeles
# ============================================================

print("=" * 60)
print("COMPARAISON FINALE DES MODELES")
print("=" * 60)

print(f"""
| Modele         | Accuracy | F1-Score | Temps/img |
|----------------|----------|----------|-----------|
| SVM + HOG      | 44.24%   | 0.4528   | ~0.1 ms   |
| CNN Custom     | 64.87%   | 0.61xx   | ~0.5 ms   |
| ResNet50       | {accuracy*100:.2f}%   | {f1_macro:.4f}  | {time_per_image:.2f} ms   |

Amelioration SVM → ResNet50 : +{(accuracy-0.4424)*100:.1f} points de precision

Analyse :
  - ResNet50 apporte un gain de {(accuracy-0.6487)*100:.1f}% par rapport au CNN Custom
  - Le transfer learning exploite les features apprises sur ImageNet
  - Le fine-tuning permet d'adapter ces features aux emotions faciales
""")

In [ ]:
# ============================================================
#  SECTION 14 : Bilan de l'etape 4c
# ============================================================

print("\n" + "=" * 60)
print("BILAN DE L'ETAPE 4c - RESNET50")
print("=" * 60)

print(f"""
Architecture :
  - Backbone : ResNet50 pre-entraine sur ImageNet (50 couches)
  - Classifieur : GlobalAveragePooling2D + Dense(512) + Dense(256) + Softmax(7)
  - Regularisation : BatchNormalization, Dropout(0.5)
  - 25 millions de parametres (backbone + classifieur)

Strategie d'entrainement :
  - Phase 1 : Classifieur uniquement (backbone gele)
    Optimiseur : Adam (lr=0.001)
    Epoques : {len(history.history['accuracy'])}
  - Phase 2 : Fine-tuning (50 dernieres couches degelees)
    Optimiseur : Adam (lr=0.0001)
    Epoques : {len(history_ft.history['accuracy'])}

Donnees :
  - Train : {train_generator.samples} images (avec data augmentation)
  - Test  : {test_generator.samples} images
  - Taille d'entree : 224x224 RGB

Resultats :
  - Accuracy : {accuracy*100:.2f}%
  - F1-Score macro : {f1_macro:.4f}
  - Temps d'inference : {time_per_image:.2f} ms/image ({fps:.1f} FPS)

Prochaines etapes :
  -> Etape 4d : VGG16 (transfer learning)
  -> Etape 4e : Ensemble (CNN Custom + ResNet50 + VGG16)
  -> Etape 5  : Interface temps reel (webcam)
""")

print("=" * 60)
print("ETAPE 4c TERMINEE")
print("=" * 60)

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, BatchNormalization, MaxPooling2D, Dropout, Flatten, Dense
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.regularizers import l2
import numpy as np
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

# Rebuild the same architecture
model = Sequential([
    Conv2D(64, (3,3), activation='relu', kernel_regularizer=l2(0.01), input_shape=(48,48,1)),
    Conv2D(64, (3,3), padding='same', activation='relu'),
    BatchNormalization(),
    MaxPooling2D(2,2),
    Dropout(0.5),
    
    Conv2D(128, (3,3), padding='same', activation='relu'),
    BatchNormalization(),
    Conv2D(128, (3,3), padding='same', activation='relu'),
    BatchNormalization(),
    MaxPooling2D(2,2),
    Dropout(0.5),
    
    Conv2D(256, (3,3), padding='same', activation='relu'),
    BatchNormalization(),
    Conv2D(256, (3,3), padding='same', activation='relu'),
    BatchNormalization(),
    MaxPooling2D(2,2),
    Dropout(0.5),
    
    Conv2D(512, (3,3), padding='same', activation='relu'),
    BatchNormalization(),
    Conv2D(512, (3,3), padding='same', activation='relu'),
    BatchNormalization(),
    MaxPooling2D(2,2),
    Dropout(0.5),
    
    Flatten(),
    Dense(512, activation='relu'),
    Dropout(0.4),
    Dense(256, activation='relu'),
    Dropout(0.4),
    Dense(128, activation='relu'),
    Dropout(0.5),
    Dense(7, activation='softmax')
])

# Load weights
model.load_weights('fer.h5')
print("Model loaded successfully!")

# Test
test_datagen = ImageDataGenerator(rescale=1./255)
test_generator = test_datagen.flow_from_directory(
    'C:/Users/ayman/Desktop/PFA/archive/test',
    target_size=(48, 48),
    color_mode='grayscale',
    class_mode='categorical',
    batch_size=64,
    shuffle=False
)

test_generator.reset()
y_pred_probs = model.predict(test_generator, steps=len(test_generator))
y_pred = np.argmax(y_pred_probs, axis=1)
y_true = test_generator.classes

accuracy = accuracy_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred, average='macro')

emotions = ['Angry', 'Disgust', 'Fear', 'Happy', 'Neutral', 'Sad', 'Surprise']

print(f"\nAccuracy: {accuracy*100:.2f}%")
print(f"F1-Score: {f1:.4f}")
print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=emotions, zero_division=0))

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=emotions, yticklabels=emotions)
plt.title('Confusion Matrix - GitHub Model')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.tight_layout()
plt.show()

TypeError: Could not locate class 'Sequential'. Make sure custom classes and functions are decorated with `@keras.saving.register_keras_serializable()`. If they are already decorated, make sure they are all imported so that the decorator is run before trying to load them. Full object config: {'class_name': 'Sequential', 'keras_version': '2.2.4', 'config': {'layers': [{'class_name': 'Conv2D', 'config': {'kernel_initializer': {'class_name': 'VarianceScaling', 'config': {'distribution': 'uniform', 'scale': 1.0, 'seed': None, 'mode': 'fan_avg'}}, 'name': 'conv2d_1', 'kernel_constraint': None, 'bias_regularizer': None, 'bias_constraint': None, 'dtype': 'float32', 'activation': 'relu', 'trainable': True, 'data_format': 'channels_last', 'filters': 64, 'padding': 'valid', 'strides': [1, 1], 'dilation_rate': [1, 1], 'kernel_regularizer': {'class_name': 'L1L2', 'config': {'l2': 0.009999999776482582, 'l1': 0.0}}, 'bias_initializer': {'class_name': 'Zeros', 'config': {}}, 'batch_input_shape': [Non